# Appendix Notebook A4: Multi-Asset Portfolio VaR and CVaR

This appendix notebook is the standardised review-paper version of the original local working file for Module 3.

This notebook extends the Module 3 framework to a multi-asset portfolio and records the comparison between classical and quantum-inspired tail-risk workflows.

## Reading Guide

The analysis covers return aggregation, portfolio loss construction, risk-metric estimation, dimensionality reduction, and interpretation of diversification effects.

All file names in `review_paper/notebooks/` follow a common naming convention so the paper appendix can reference them consistently.

#  3.6.2 Multi-Asset Portfolio VaR/CVaR  
### Classical Baseline + Quantum-Inspired PCA Simulation

This notebook implements Section **3.6.2** of *Module 3: Quantum Risk Estimation and Scenario Simulation*.

We compute portfolio tail-risk under three methods:

---

## 1. Classical Historical VaR/CVaR
Using a fixed two-year daily panel stored as local Yahoo Finance CSV snapshots (1 January 2024 to 31 December 2025), we compute:

- Daily log-returns for each asset  
- Portfolio daily loss distribution  
- Historical VaR (quantile)  
- Historical CVaR (tail mean)

Mathematically:

$ \text{VaR}_{\alpha} = \inf \{ x \mid P(L \le x) \ge \alpha \} $

$ \text{CVaR}_{\alpha} = \mathbb{E}[ L \mid L \ge \text{VaR}_{\alpha} ] $

---

## 2. Parametric (Normal) VaR/CVaR
Assuming portfolio loss is approximately normal:

$ L \sim N(\mu_p, \sigma_p^2) $

$ \text{VaR}_{\alpha}^{\text{norm}} = \mu_p + \sigma_p \Phi^{-1}(\alpha) $

$ \text{CVaR}_{\alpha}^{\text{norm}} 
= \mu_p + \sigma_p \frac{\phi(\Phi^{-1}(\alpha))}{1-\alpha} $

---

## 3. Quantum-Inspired PCA Scenario Simulation
To mimic "quantum loading of multivariate distributions", we:

1. Estimate asset-return covariance  
2. Perform **PCA eigen-decomposition**  
3. Generate synthetic correlated samples:

$ R_{\text{sim}} = \mu + LZ, \quad Z \sim N(0, I_k) $

where  
- $L$ = PCA factor loading matrix  
- $Z$ = independent Gaussian shocks  

This acts as a **quantum-inspired generative model**, approximating quantum probability loading.

We then compute VaR/CVaR on the PCA-generated distribution.

---

##  Output
The notebook will print:

- Historical VaR/CVaR  
- Normal-parametric VaR/CVaR  
- PCA (quantum-inspired) VaR/CVaR  
- Comparison results

Optionally, we can add:

- Plots of PCA factor loadings  
- Synthetic scenario clouds  
- Portfolio loss distribution overlays  

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from pathlib import Path

def locate_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'review_paper' / 'assets').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root from the current working directory.')


def load_local_close_series(csv_path: Path, ticker: str) -> pd.Series:
    raw = pd.read_csv(csv_path, header=[0, 1])
    dates = pd.to_datetime(raw.iloc[1:, 0])
    closes = pd.to_numeric(raw.iloc[1:, 1], errors='coerce')
    return pd.Series(closes.to_numpy(), index=dates, name=ticker).dropna()

ROOT = locate_repo_root(Path.cwd())
ASSET_DATA_DIR = ROOT / 'review_paper' / 'assets' / 'module_03_risk' / 'data'
FIG_PATH = ROOT / 'review_paper' / 'multi_asset_loss_plot.pdf'
ASSET_DATA_DIR.mkdir(parents=True, exist_ok=True)

import warnings
warnings.filterwarnings('ignore')


##### ==============================
##### 0. Parameter setup
##### ==============================

In [ ]:
TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN']  # Multi-asset universe
ALPHA   = 0.95                                # Confidence level for VaR/CVaR
HORIZON_DAYS  = 1                             # 1-day horizon
START_DATE = '2024-01-01'
END_DATE = '2025-12-31'
SEED = 42
np.random.seed(SEED)

# Portfolio weights (must sum to 1)
# Example: equal-weight portfolio
weights = np.array([1.0 / len(TICKERS)] * len(TICKERS))

print('Universe:', TICKERS)
print('Portfolio weights:', weights)


##### ==============================
##### 1. Download data & construct returns
##### ==============================

In [ ]:
csv_paths = [ASSET_DATA_DIR / f'{ticker}_{START_DATE}_{END_DATE}_daily.csv' for ticker in TICKERS]
missing = [str(path) for path in csv_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Missing expected local CSVs: ' + ', '.join(missing))

print(f'\nLoading local market data from {START_DATE} to {END_DATE} (inclusive) ...')
prices = pd.concat(
    [load_local_close_series(path, ticker) for ticker, path in zip(TICKERS, csv_paths)],
    axis=1,
).sort_index().dropna()
print('Price data shape:', prices.shape)

# Daily log returns
rets = np.log(prices / prices.shift(1)).dropna()
print('Return data shape:', rets.shape)

# Align weights with columns
if len(weights) != rets.shape[1]:
    raise ValueError('Length of weights does not match number of assets.')

# Portfolio returns: r_p = w^T r
ret_port_hist = rets.values @ weights
losses_hist   = -ret_port_hist  # define loss = -return (positive = loss)

n_hist = len(losses_hist)
print(f'Sample size (daily portfolio returns): {n_hist}')


##### ==============================
##### 2. Classical baseline: portfolio VaR & CVaR
##### ==============================

In [ ]:
# 2.1 Historical VaR / CVaR
var_hist = np.quantile(losses_hist, ALPHA)

tail_losses_hist = losses_hist[losses_hist >= var_hist]
cvar_hist = tail_losses_hist.mean()

# 2.2 Parametric (multivariate normal) VaR / CVaR
# Estimate mean vector and covariance matrix
mu_vec    = rets.mean().values              # mean of each asset
Sigma_hat = rets.cov().values               # covariance matrix

# Portfolio mean and variance under multivariate normal
mu_p    = float(weights @ mu_vec)
var_p   = float(weights @ Sigma_hat @ weights)   # portfolio variance
sigma_p = np.sqrt(var_p)

z_alpha  = norm.ppf(ALPHA)

# Portfolio VaR and CVaR (normal approximation)
var_norm  = - mu_p + sigma_p * z_alpha  # note: VaR in "loss" sign convention
phi_alpha = norm.pdf(z_alpha)
cvar_norm = - mu_p + sigma_p * (phi_alpha / (1 - ALPHA))

print("\n=== Classical Baseline (1-day, multi-asset portfolio) ===")
print(f"Alpha = {ALPHA:.2%}")
print(f"[Historical]     VaR  = {var_hist:.4%},   CVaR = {cvar_hist:.4%}")
print(f"[Normal (param)] VaR  = {var_norm:.4%},   CVaR = {cvar_norm:.4%}")

##### ==============================
##### 3. PCA-based generative model (quantum-inspired multivariate loading)
##### ==============================

In [ ]:
# Idea:
#   - Estimate covariance matrix Sigma_hat.
#   - Perform PCA: Sigma_hat  V  V^T.
#   - Keep the top K_FACTORS principal components.
#   - Generate correlated samples r_sim ~ N(mu_vec, Sigma_PCA).
#   - This mimics a low-rank quantum generative model over a multivariate state.

N_SIM     = 100_000        # Number of simulated scenarios
K_FACTORS = min(3, len(TICKERS))  # Number of PCA factors (e.g., 2 or 3)

# Eigen-decomposition of covariance matrix
eigvals, eigvecs = np.linalg.eigh(Sigma_hat)
# Sort eigenvalues / eigenvectors in descending order
idx_sort = np.argsort(eigvals)[::-1]
eigvals  = eigvals[idx_sort]
eigvecs  = eigvecs[:, idx_sort]

# Take top K_FACTORS components
Lambda_k = np.diag(np.sqrt(eigvals[:K_FACTORS]))
V_k      = eigvecs[:, :K_FACTORS]     # shape: (n_assets, K_FACTORS)

# Factor loading matrix: L = V_k * sqrt(Lambda_k)
L_pca = V_k @ Lambda_k                # shape: (n_assets, K_FACTORS)

# Generate N_SIM samples of K_FACTORS standard normal factors
Z = np.random.randn(N_SIM, K_FACTORS)

# Generate correlated returns: r_sim = mu_vec + Z * L_pca^T
r_sim = Z @ L_pca.T + mu_vec          # shape: (N_SIM, n_assets)

# Portfolio simulated returns and losses
ret_port_sim = r_sim @ weights
losses_sim   = -ret_port_sim

print(f"\nPCA-based generative model:")
print(f"Eigenvalues (descending): {eigvals}")
print(f"Using top {K_FACTORS} factors for simulation.")
print(f"Simulated scenarios: {N_SIM}")

##### ==============================
##### 4. Quantum-inspired VaR/CVaR via histogram + threshold search
#####    (on PCA-simulated portfolio loss distribution)
##### ==============================

In [ ]:
N_BINS = 64  # number of bins for discretisation (analogous to 6 qubits)

# 4.1 Histogram discretisation
hist_counts, bin_edges = np.histogram(losses_sim, bins=N_BINS, density=False)
p = hist_counts / hist_counts.sum()      # probability vector
bin_centers = 0.5 * (bin_edges[1:] + bin_edges[:-1])

# 4.2 Threshold search for VaR equivalent:
#     For each candidate threshold L_cand (on the grid), compute:
#         P(loss >= L_cand) = sum_{i: center_i >= L_cand} p_i
#     Find L_cand such that this tail probability is closest to (1 - ALPHA).

target_tail_prob = 1.0 - ALPHA
candidate_VaRs = bin_centers

tail_probs = []
for L_cand in candidate_VaRs:
    mask = (bin_centers >= L_cand)
    tail_prob = p[mask].sum()
    tail_probs.append(tail_prob)

tail_probs = np.array(tail_probs)

idx_best = np.argmin(np.abs(tail_probs - target_tail_prob))
var_qi   = candidate_VaRs[idx_best]
tail_prob_best = tail_probs[idx_best]

# 4.3 Quantum-inspired CVaR:
#     Conditional expectation over the tail region in the discretised distribution
mask_tail_bins = (bin_centers >= var_qi)
p_tail = p[mask_tail_bins]
L_tail = bin_centers[mask_tail_bins]

if p_tail.sum() > 0:
    cvar_qi = (p_tail * L_tail).sum() / p_tail.sum()
else:
    cvar_qi = var_qi  # fallback in degenerate case

print("\n=== Quantum-inspired (PCA generative model + histogram threshold) ===")
print(f"Target tail prob: 1 - alpha = {target_tail_prob:.2%}")
print(f"Chosen VaR (grid)    = {var_qi:.4%}")
print(f"Tail prob at VaR     = {tail_prob_best:.2%}")
print(f"CVaR (grid tail mean) = {cvar_qi:.4%}")

##### ==============================
##### 5. Summary comparison
##### ==============================

In [ ]:
print("\n=== Summary (1-day portfolio loss %) ===")
print(f"[Historical] VaR  : {var_hist*100:.3f}%,   CVaR: {cvar_hist*100:.3f}%")
print(f"[Normal]     VaR  : {var_norm*100:.3f}%,   CVaR: {cvar_norm*100:.3f}%")
print(f"[QI-PCA]     VaR  : {var_qi*100:.3f}%,   CVaR: {cvar_qi*100:.3f}%")

##### ==============================
##### 5. Plot: portfolio loss histogram + VaR/CVaR lines
##### ==============================

In [ ]:
# ==============================
# 5. Plot: portfolio loss histogram + VaR/CVaR lines (color-grouped)
# ==============================

# Convert losses to percentage
losses_hist_pct = losses_hist * 100
losses_sim_pct  = losses_sim * 100

var_hist_pct  = var_hist * 100
cvar_hist_pct = cvar_hist * 100
var_norm_pct  = var_norm * 100
cvar_norm_pct = cvar_norm * 100
var_qi_pct    = var_qi * 100
cvar_qi_pct   = cvar_qi * 100

plt.figure(figsize=(10, 6))

# -----------------------------------
# Histograms
# -----------------------------------
# Histogram of historical portfolio losses
plt.hist(
    losses_hist_pct,
    bins=50,
    density=True,
    alpha=0.5,
    label='Historical portfolio loss'
)

# Histogram of synthetic (QI-PCA) losses for visual comparison
plt.hist(
    losses_sim_pct,
    bins=50,
    density=True,
    alpha=0.3,
    label='QI-PCA synthetic loss'
)

# -----------------------------------
# Historical VaR/CVaR  RED group
# -----------------------------------
plt.axvline(
    var_hist_pct,
    color='red',
    linestyle='--',
    linewidth=2,
    label=f'Historical VaR ({var_hist_pct:.2f}%)'
)
plt.axvline(
    cvar_hist_pct,
    color='red',
    linestyle=':',
    linewidth=2,
    label=f'Historical CVaR ({cvar_hist_pct:.2f}%)'
)

# -----------------------------------
# Normal (parametric) VaR/CVaR  BLUE group
# -----------------------------------
plt.axvline(
    var_norm_pct,
    color='blue',
    linestyle='--',
    linewidth=2,
    label=f'Normal VaR ({var_norm_pct:.2f}%)'
)
plt.axvline(
    cvar_norm_pct,
    color='blue',
    linestyle=':',
    linewidth=2,
    label=f'Normal CVaR ({cvar_norm_pct:.2f}%)'
)

# -----------------------------------
# QI-PCA VaR/CVaR  PURPLE group
# -----------------------------------
plt.axvline(
    var_qi_pct,
    color='purple',
    linestyle='--',
    linewidth=2,
    label=f'QI-PCA VaR ({var_qi_pct:.2f}%)'
)
plt.axvline(
    cvar_qi_pct,
    color='purple',
    linestyle=':',
    linewidth=2,
    label=f'QI-PCA CVaR ({cvar_qi_pct:.2f}%)'
)

# -----------------------------------
# Formatting
# -----------------------------------
plt.title('Multi-asset portfolio loss distribution with VaR/CVaR')
plt.xlabel('Loss (%)')
plt.ylabel('Density')
plt.grid(alpha=0.3)
plt.legend()

# Save in PDF format
out_file = FIG_PATH
plt.savefig(out_file, format='pdf', bbox_inches='tight')
print('Saved plot to:', out_file.resolve())

plt.show()
plt.close()


## Interpretation Note

This notebook is preserved in the appendix asset set so the review paper can keep executable detail separate from the main argument.

In [ ]:
summary = pd.DataFrame([
    {
        'method': 'Historical',
        'var_1d_pct': var_hist * 100,
        'cvar_1d_pct': cvar_hist * 100,
        'notes': 'empirical portfolio loss quantile',
    },
    {
        'method': 'Normal',
        'var_1d_pct': var_norm * 100,
        'cvar_1d_pct': cvar_norm * 100,
        'notes': 'multivariate normal approximation',
    },
    {
        'method': 'Quantum-inspired PCA',
        'var_1d_pct': var_qi * 100,
        'cvar_1d_pct': cvar_qi * 100,
        'notes': 'PCA-based correlated scenario model',
    },
])
summary_path = ASSET_DATA_DIR / 'module_03_multi_asset_var_cvar_summary.csv'
summary.to_csv(summary_path, index=False, float_format='%.3f')
display(summary)
print(f'Summary saved to: {summary_path}')
